<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/02_demand_eda_single_product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 02 · Demand EDA — Single Product

**Data Science · Università Cattolica del Sacro Cuore, Milano**
Instructor: *Paolo Balasso*

---

## What you will explore

A complete EDA + baseline-forecasting workflow on a **single SKU** daily demand series, building the foundation before any complex forecasting model.

1. Generate a realistic synthetic daily sales series (trend + weekly + yearly + noise + outliers)
2. **Decomposition** — trend / seasonality / residual (additive STL)
3. **Stationarity** — Augmented Dickey-Fuller test (light touch)
4. **Autocorrelation** — ACF / PACF
5. **Outlier handling** — rolling IQR, replacement strategies
6. **Baseline forecasts** — Naive, Seasonal-Naive, Holt-Winters (ETS), linear trend
7. **Accuracy metrics** — MAPE / WAPE / MASE (slide recap + interpretation)
8. **Residual diagnostics** — what does the model fail to capture?

**Style.** Short readable code, lots of plots. Mini-exercises are *parameter tweaks* — change seasonality, change holdout, observe.

---

## 0 · Setup

In [ ]:
%pip install -q statsmodels matplotlib seaborn pandas numpy scipy

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

NAVY, GOLD, INK, MUTED = '#1B263B', '#D4A017', '#0F141E', '#556070'
GREEN, RED = '#2D6E3E', '#B32D2D'
sns.set_style('whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK})
np.random.seed(42)
print('Setup complete.')

## 1 · A realistic synthetic SKU

We build a daily-frequency series that mimics what you would see in a retailer's POS: positive trend, weekly seasonality (weekend spike), yearly seasonality (Q4 ramp-up), Gaussian noise, plus a *promo spike* and a *stockout dip* as planted outliers.

In [ ]:
def make_sku(start='2022-01-01', n_days=730, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start, periods=n_days, freq='D')
    t = np.arange(n_days)

    trend     = 100 + 0.06*t                                      # +6 units/100 days
    weekly    = 12*np.sin(2*np.pi*(t % 7)/7 - 1.0)                # weekend peak
    yearly    = 25*np.sin(2*np.pi*(t % 365)/365 - 1.8)            # Q4 ramp
    noise     = rng.normal(0, 6, n_days)
    series    = trend + weekly + yearly + noise

    # Plant a promo spike (5 days) and a stockout dip (3 days)
    promo_start, promo_len = 300, 5
    series[promo_start:promo_start+promo_len] += 60               # spike
    stockout_start, stockout_len = 480, 3
    series[stockout_start:stockout_start+stockout_len] *= 0.10    # dip

    return pd.Series(series.clip(min=0), index=dates, name='units_sold')

y = make_sku()
print(f'SKU series: {len(y)} days, from {y.index.min().date()} to {y.index.max().date()}')
print(f'Mean: {y.mean():.1f}, Std: {y.std():.1f}, Min: {y.min():.1f}, Max: {y.max():.1f}')
y.head()

In [ ]:
# Eyeball: full series + zoom on the planted anomalies
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
axes[0].plot(y.index, y.values, color=NAVY, lw=0.9)
axes[0].set_title('Daily units sold (full sample)',
                   fontweight='bold', color=INK)
axes[0].axvspan(y.index[300], y.index[305], color=GREEN, alpha=0.25, label='Promo')
axes[0].axvspan(y.index[480], y.index[483], color=RED,   alpha=0.25, label='Stockout')
axes[0].legend(loc='upper left')

# Zoom on October-December to show yearly seasonality
zoom = y.loc['2022-10-01':'2022-12-31']
axes[1].plot(zoom.index, zoom.values, color=NAVY, lw=1.1)
axes[1].set_title('Zoom: Q4 2022 (weekly cycle visible)',
                   fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

## 2 · STL decomposition — separating signal from noise

STL = Seasonal-Trend decomposition using LOESS. Robust, handles outliers, lets us see each component independently.

In [ ]:
# Weekly seasonality (period=7) since daily data
stl = STL(y, period=7, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
parts = [
    ('Observed',   y,             NAVY),
    ('Trend',      stl.trend,     GOLD),
    ('Seasonal',   stl.seasonal,  GREEN),
    ('Residual',   stl.resid,     RED),
]
for ax, (label, series, color) in zip(axes, parts):
    ax.plot(series.index, series.values, color=color, lw=0.9)
    ax.set_title(label, fontweight='bold', color=INK, fontsize=10, loc='left')
plt.suptitle('STL decomposition (period = 7 days)',
             fontweight='bold', color=INK, y=1.005)
plt.tight_layout(); plt.show()
print('\nResidual std:', round(stl.resid.std(), 2),
      '— this is the noise floor any model has to beat.')

> **Mini-exercise (parametric).** Re-run with `period=14` and `period=30`. Note how the seasonal component becomes meaningless when the period does not match the data's true cycle — the structure leaks into the residual. STL is *not* finding the period for you.

## 3 · Stationarity check (light touch)

Many forecasting methods assume the series is **stationary** (constant mean and variance). The **Augmented Dickey-Fuller (ADF)** test: H0 = "the series has a unit root" (non-stationary). Small p-value → reject → stationary.

In [ ]:
def adf_summary(series, label):
    stat, p, *_ = adfuller(series.dropna(), autolag='AIC')
    verdict = 'STATIONARY (reject H0)' if p < 0.05 else 'NON-stationary'
    print(f'{label:30s}  ADF stat={stat:7.3f}  p={p:.4f}   → {verdict}')

adf_summary(y, 'Raw series')
adf_summary(y.diff().dropna(), '1st differences')
adf_summary(stl.resid, 'STL residual')

**Read-out.** The raw series is usually non-stationary (trend present). First differences and STL residuals are typically stationary. This shapes which model class is appropriate (ARIMA needs differencing, ETS handles trend natively).

## 4 · Autocorrelation — ACF & PACF

- **ACF**: correlation of the series with its own lags. Tells you how far in the past the signal still echoes.
- **PACF**: same but *partialing out* shorter lags. Diagnostic for AR-order selection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(y, lags=30, ax=axes[0], color=NAVY)
axes[0].set_title('ACF — daily sales', fontweight='bold', color=INK)
plot_pacf(y, lags=30, ax=axes[1], method='ywm', color=NAVY)
axes[1].set_title('PACF — daily sales', fontweight='bold', color=INK)
plt.tight_layout(); plt.show()
print('Look for spikes at lag 7, 14, 21 → weekly seasonality.')
print('Slow ACF decay → trend not yet removed.')

## 5 · Outlier detection & treatment

Outliers fall into two business categories:
- **Real events to remember** (promo, holiday) — keep them but flag them
- **Data errors** (stockout, scanner glitch) — replace with a robust estimate

We use a **rolling-window IQR rule** as a transparent baseline detector.

In [ ]:
def detect_outliers_iqr(series, window=14, k=2.5):
    rolling = series.rolling(window, center=True, min_periods=5)
    q1 = rolling.quantile(0.25); q3 = rolling.quantile(0.75)
    iqr = q3 - q1
    low  = q1 - k*iqr; high = q3 + k*iqr
    is_out = (series < low) | (series > high)
    return is_out, low, high

is_out, low, high = detect_outliers_iqr(y, window=14, k=2.5)
print(f'Flagged outliers: {is_out.sum()} ({is_out.mean():.1%} of days)')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y.index, y.values, color=NAVY, lw=0.7, label='Observed')
ax.fill_between(y.index, low, high, color=GOLD, alpha=0.2, label='IQR band (k=2.5)')
ax.scatter(y.index[is_out], y.values[is_out], color=RED, s=18, zorder=5, label='Outlier')
ax.set_title('Outlier detection — rolling IQR rule',
             fontweight='bold', color=INK)
ax.legend(loc='upper left'); plt.tight_layout(); plt.show()

In [ ]:
# Replace flagged outliers with rolling median (a conservative choice)
y_clean = y.copy()
y_clean[is_out] = y.rolling(7, center=True, min_periods=3).median()[is_out]
y_clean = y_clean.fillna(y.median())
print(f'Mean before:  {y.mean():.2f}')
print(f'Mean after:   {y_clean.mean():.2f}  (small shift, robust treatment)')

> **Mini-exercise (parametric).** Change `k=2.5` to `k=1.5` (more aggressive) and `k=4.0` (more permissive). Re-run. Outlier handling is a **design choice**, not a single right answer — the trade-off is between losing real signal (over-cleaning) and contaminating the model (under-cleaning).

## 6 · Baseline forecasts

Before touching ARIMA or Prophet or a neural net, you owe these four baselines:

1. **Naive** — tomorrow = today
2. **Seasonal-Naive (SNaive)** — tomorrow = today *7 days ago*
3. **Linear trend** — extrapolate a regression line
4. **Holt-Winters (ETS)** — exponential smoothing with trend & seasonality

If a fancy model cannot beat these four, it has no business being in production.

In [ ]:
# Train/test split: last 60 days as holdout
H = 60
train, test = y_clean.iloc[:-H], y_clean.iloc[-H:]
print(f'Train: {len(train)} days   Test: {len(test)} days')
print(f'Test window: {test.index.min().date()} → {test.index.max().date()}')

In [ ]:
# Forecasts
fc = pd.DataFrame(index=test.index)
# Naive
fc['Naive']  = train.iloc[-1]
# Seasonal-Naive (weekly)
fc['SNaive'] = train.iloc[-7 - (np.arange(H) % 7)].values
# Linear trend
t_tr = np.arange(len(train)); t_te = np.arange(len(train), len(train)+H)
slope, intercept = np.polyfit(t_tr, train.values, 1)
fc['LinearTrend'] = slope*t_te + intercept
# Holt-Winters
hw = ExponentialSmoothing(train, trend='add', seasonal='add',
                          seasonal_periods=7).fit()
fc['Holt-Winters'] = hw.forecast(H).values
fc.head()

In [ ]:
# Plot forecasts vs actuals on the holdout
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train.index[-90:], train.values[-90:], color=MUTED, lw=0.8, label='Train (last 90d)')
ax.plot(test.index, test.values, color=NAVY, lw=1.5, label='Actual')
for col, color in zip(['Naive','SNaive','LinearTrend','Holt-Winters'],
                      [RED, GOLD, GREEN, '#7B2CBF']):
    ax.plot(fc.index, fc[col], color=color, lw=1.0, ls='--', label=col)
ax.set_title('Holdout forecasts — last 60 days',
             fontweight='bold', color=INK)
ax.legend(loc='upper left', ncol=3, fontsize=8)
plt.tight_layout(); plt.show()

## 7 · Accuracy metrics — MAPE / WAPE / MASE

Three classic forecasting metrics with very different sensitivities:

- **MAPE** = mean of |error| / |actual|. Easy to communicate. **Explodes** when actuals are near zero (intermittent demand → unusable).
- **WAPE** = sum |error| / sum |actual|. The volume-weighted version. **Robust to zeros**. Industry standard for retail.
- **MASE** = mean |error| / mean |SNaive error|. **Scale-free**. MASE < 1 means you beat the seasonal-naive baseline.

In [ ]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))

def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

def mase(y_true, y_pred, y_train, season=7):
    naive_err = np.mean(np.abs(np.diff(y_train, n=season)))
    return np.mean(np.abs(y_true - y_pred)) / naive_err

metrics = pd.DataFrame(
    {col: {
        'MAPE': mape(test.values, fc[col].values),
        'WAPE': wape(test.values, fc[col].values),
        'MASE': mase(test.values, fc[col].values, train.values),
    } for col in fc.columns}
).round(4)
metrics

**Read-outs.**
- MASE < 1 → the model beats Seasonal-Naive (by definition the benchmark).
- WAPE is the cleanest cross-method comparison; communicate it to business.
- A method that "wins" on MAPE but "loses" on WAPE is being inflated by errors at small actuals — interrogate it.

> **Mini-exercise (parametric).** Change `H = 60` to `H = 30` (shorter holdout, less generalisation evidence) and to `H = 120` (longer holdout). Watch how the ranking of models can change. Forecast quality is *not* a constant; it depends on horizon and window.

## 8 · Residual diagnostics — what does the best model still miss?

If residuals look like white noise, you have extracted the signal. If they show structure, there is *more model* to build.

In [ ]:
resid = test.values - fc['Holt-Winters'].values
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(test.index, resid, color=NAVY, lw=0.9)
axes[0].axhline(0, color=MUTED, ls='--', lw=0.7)
axes[0].set_title('Holt-Winters residuals over time',
                   fontweight='bold', color=INK, fontsize=10)

axes[1].hist(resid, bins=20, color=NAVY, alpha=0.7, edgecolor='white')
axes[1].axvline(0, color=GOLD, ls='--', lw=0.8)
axes[1].set_title('Residual distribution',
                   fontweight='bold', color=INK, fontsize=10)

plot_acf(resid, lags=14, ax=axes[2])
axes[2].set_title('Residual ACF (lags 1-14)',
                   fontweight='bold', color=INK, fontsize=10)
plt.tight_layout(); plt.show()
print(f'Residual mean: {resid.mean():.2f}  (should be ~0 if unbiased)')
print(f'Residual std:  {resid.std():.2f}')

## 9 · Caveats & next steps

1. **Synthetic data flatters models.** Real demand series have promotions, holidays, supply disruptions, price changes — exogenous regressors none of the four baselines use.
2. **Single point forecast hides uncertainty.** Always produce **prediction intervals** (Holt-Winters gives them for free via `simulations` or formulas).
3. **Recent observations matter more.** Re-fit weekly in production; do not freeze a model from 6 months ago.
4. **MAPE is the favourite metric of people who do not know better.** Default to WAPE for sums, MASE for cross-series comparison.

### Where to go next

- Add exogenous regressors: price, promotion flag, weather → SARIMAX or Prophet
- Move to **probabilistic forecasts**: quantile regression, conformal intervals
- For the multi-product case (gerarchia, mix problem, riconciliazione) → notebook **03_demand_eda_multi_product**

### References

- Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* (3rd ed.) — open access at otexts.com/fpp3
- Makridakis, Spiliotis, Assimakopoulos (2018). *Statistical and ML forecasting methods: concerns and ways forward*. PLoS One.
- Slides Main FIX8 §5 — Forecast Accuracy + Ensemble (slides 26–28).

---

**End of notebook.** ~15 seconds of runtime on Colab CPU.